# run_all_0416b_nb9 — two-stage adaptive seller, switch at round 11

Part of the 0416b negotiation-data sweep (deadline-free version).

* Seller strategies covered : **8 two-stage seller schedules, switch @ round 11 of 15**
* Buyer strategies covered  : **all 10 fixed buyers**
* Mode                      : **adaptive seller × fixed buyer**
* Target per-notebook runtime : **~18 h** on `deepseek-reasoner`
* No round-limit or deadline information is included in any prompt.

The six seller strategies in the experiment universe are:
`anchor_high`, `cooperative`, `boulware_seller`, `conceder_seller`,
`tit_for_tat_matcher`, `relational`.

The ten buyer strategies are: `random`, `budget`, `wild`, `concession`,
`anchor_drag`, `boulware_buyer`, `conceder_buyer`, `tit_for_tat_buyer`,
`persistent_f0.30`, `persistent_f0.55`.


In [1]:
import os
# ── Imports ───────────────────────────────────────────────────────────
import sys
from pathlib import Path

HERE = Path.cwd()
for p in (HERE, HERE.parent, HERE.parent.parent):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

from openai import OpenAI

import exp_runner_0416 as R
from run_all_0416 import run_experiment_block

from exp_runner_0416 import (
    RandomBuyer, BudgetBuyer, WildBuyer,
    ConcessionBuyer, AnchorDragBuyer,
    BoulwareBuyer, ConcederBuyer, TitForTatBuyer,
    PersistentBuyer, AdaptiveBuyer,
    fixed_seller_schedule, twostage_seller_schedule, threestage_seller_schedule,
)
from run_all_0416 import (
    all_fixed_buyers_for_notebook, fixed_pairs_for_sellers,
)


# ── Config (edit as needed) ──────────────────────────────────────────
CLIENT = OpenAI(
    api_key=os.environ["DEEPSEEK_API_KEY"],  # Set this key in your environment.
    base_url="https://api.deepseek.com",
)
MODEL       = "deepseek-reasoner"
N_ROUNDS    = 15
BASE_SEED   = 5000 + 9000        # distinct seed offset per notebook
OUTPUT_ROOT = Path("results_0416b")
NOTEBOOK_ID = "nb9"


## EXPERIMENTS list — notebook 9

In [2]:
# 8 two-stage seller schedules covering hard→soft, soft→hard,
# and opposite-philosophy transitions among the 6 base strategies.
TWO_STAGE_PAIRS = [
    ("anchor_high",        "cooperative"),
    ("anchor_high",        "conceder_seller"),
    ("boulware_seller",    "conceder_seller"),
    ("boulware_seller",    "cooperative"),
    ("cooperative",        "anchor_high"),
    ("conceder_seller",    "boulware_seller"),
    ("tit_for_tat_matcher","anchor_high"),
    ("relational",         "boulware_seller"),
]

SWITCH_ROUND     = 11
N_RUNS_PER_PAIR  = 3

BUYERS = all_fixed_buyers_for_notebook()

EXPERIMENTS = []
for first, second in TWO_STAGE_PAIRS:
    sched = twostage_seller_schedule(first, second, switch_round=SWITCH_ROUND,
                                     n_rounds=N_ROUNDS)
    for b in BUYERS:
        tag = f"sel2_{first}__{second}__r{SWITCH_ROUND}__buy_{b.name}"
        EXPERIMENTS.append({
            "tag": tag, "buyer": b, "seller_schedule": sched,
            "n_runs": N_RUNS_PER_PAIR,
        })

print(f"Total conditions: {len(EXPERIMENTS)}")
print(f"Total runs      : {sum(e['n_runs'] for e in EXPERIMENTS)}")


Total conditions: 80
Total runs      : 240


## Run the block

In [3]:
# ── Execute ───────────────────────────────────────────────────────────
manifest = run_experiment_block(
    experiments=EXPERIMENTS,
    notebook_id=NOTEBOOK_ID,
    client=CLIENT,
    model=MODEL,
    n_rounds=N_ROUNDS,
    base_seed=BASE_SEED,
    output_root=OUTPUT_ROOT,
    verbose=True,
    resume=True,   # skip conditions whose JSON already exists
)
print("\n--- Per-condition timings (minutes) ---")
for t in manifest["timings"]:
    print(t)



[1/80]  sel2_anchor_high__cooperative__r11__buy_random
  buyer   = random
  seller  = anchor_high__then__cooperative@r11  (adaptive=True)
  n_runs  = 3   n_rounds = 15

  --- Run 1/3  seed=14000  prod=Chocolate bar (45g) cost=$0.67 mkt=$2.85 int=$0.93 ---
  R 1 BUY $0.97 T3 [buy=random]  SEL $2.80 [sel=anchor_high]  "Thank you for your interest. Given the high quality of our c..."
  R 2 BUY $1.09 T3 [buy=random]  SEL $2.75 [sel=anchor_high]  "Thank you for improving your offer, but $1.09 per unit still..."
  R 3 BUY $2.35 T4 [buy=random]  SEL $2.70 [sel=anchor_high]  "Thank you for your improved offer of $2.35. While it is clos..."
  R 4 BUY $2.46 T0 [buy=random]  SEL $2.65 [sel=anchor_high]  "I appreciate your offer of $2.46, which shows you're moving ..."
  R 5 BUY $1.89 T3 [buy=random]  SEL $2.60 [sel=anchor_high]  "Thank you for your offer. While $1.89 per unit shows movemen..."
  R 6 BUY $1.67 T0 [buy=random]  SEL $2.55 [sel=anchor_high]  "Thank you for your continued interest. H

## Inspection

In [4]:
# ── Quick inspection of the first run of the first condition ────────
import json
first_tag = EXPERIMENTS[0]["tag"]
with open(OUTPUT_ROOT / NOTEBOOK_ID / f"{first_tag}.json") as f:
    exp = json.load(f)
R.inspect_run(exp["runs"][0], max_chars=200)


UnicodeDecodeError: 'gbk' codec can't decode byte 0x94 in position 3796: illegal multibyte sequence